# Predicting daily bike demand over time

Automated execution snapshot · human review pending.

Can calendar and observed weather variables estimate demand in a later time period?

The findings below were computed by the agent. Code cells are intentionally unexecuted in this exported walkthrough; run them to inspect and reproduce the study.

## Recorded findings

Ridge regression was selected using training cross-validation. Its holdout rmse was 1165.6547, versus 2560.5558 for the dummy baseline; it improved on that baseline on this holdout.

Training rmse was 800.9383. The training/holdout difference is descriptive; it is not an independent estimate of model uncertainty.

atemp had the largest mean permutation score drop (326.6879). This measures the fitted model's reliance on a feature, not a causal effect; correlated features can share importance.

The largest holdout absolute error was 4255.6062 target units. Error examples are retained in error_analysis.csv.

The strongest absolute Pearson feature correlation in the training data was temp / atemp (|r| = 0.997); this suggests checking redundancy, not concluding causality.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd

project = Path.cwd()
if not (project / 'project.json').exists():
    project = project / 'projects' / '2026-09-05-bike-demand'
repo = project.parents[1]
sys.path.insert(0, str(repo))
from daily_ds.reproduce import load_snapshot
manifest = json.loads((project / 'project.json').read_text())
frame = load_snapshot(project, manifest)
frame.head()


## Data quality

Explain the row unit, missing values, duplicates and excluded columns before interpreting model scores.

In [ ]:
pd.read_csv(project / 'data_dictionary.csv')


## Evaluation and reproduction

Preprocessing is fitted within training folds. Hyperparameters are fixed before CV, and only the CV winner and baseline touch the holdout. Re-running against the same holdout is reproducibility, not new evidence.

In [ ]:
from daily_ds.reproduce import reproduce
# Re-runs from the saved snapshot and config, writing to reproduced/.
reproduce(project, repo / 'reproduced' / project.name)


## Your review

Observed same-day weather may be unavailable at forecast time, so this is a retrospective conditional demand estimate. The final chronological holdout crosses seasons and can expose distribution shift. Casual and registered counts are excluded because they sum to the target.

Fill LEARNING_NOTES.md in your own words. Propose one change, justify it, and evaluate it with a fresh test protocol.